# 🛠️ Ingestão de Dados Brutos — Camada Bronze (SISCAN / 10ª CRS)

Este notebook é responsável pela **etapa de ingestão e estruturação inicial** dos dados públicos sobre exames citopatológicos do colo do útero (SISCAN) na **Camada Bronze** da Arquitetura Medallion.

---

## 1. Descompactação e Organização dos Arquivos Fonte

**Objetivo:** Extrair a estrutura de diretórios do arquivo `dataset.zip` e salvá-la no diretório de destino do Volume no Unity Catalog (`siscan_files_bronze/dataset/`), garantindo a preservação da hierarquia original sem duplicação de pastas.

* **Fonte:** `/Volumes/tcc-analise-siscan-10crs/bronze/siscan_files_bronze/dataset.zip`
* **Destino:** `/Volumes/tcc-analise-siscan-10crs/bronze/siscan_files_bronze/dataset/`

In [0]:
import zipfile
import shutil
import os

# Caminho do volume e do ficheiro zip
volume_path = "/Volumes/tcc-analise-siscan-10crs/bronze/siscan_files_bronze"
zip_path = f"{volume_path}/dataset.zip"
target_dataset_dir = f"{volume_path}/dataset"
temp_extract_dir = f"{volume_path}/temp_extraicao"

# 1. Garante que a pasta 'dataset' exista
os.makedirs(target_dataset_dir, exist_ok=True)

# 2. Extrai o zip numa pasta temporária
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(temp_extract_dir)

# 3. Se dentro do zip já existia uma pasta 'dataset', move o conteúdo dela diretamente;
# caso contrário, move o conteúdo da extração temporária
nested_dir = os.path.join(temp_extract_dir, "dataset")
source_dir = nested_dir if os.path.exists(nested_dir) else temp_extract_dir

for item in os.listdir(source_dir):
    src = os.path.join(source_dir, item)
    dst = os.path.join(target_dataset_dir, item)
    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.move(src, dst)
    else:
        shutil.move(src, dst)

# 4. Remove a pasta temporária de trabalho
if os.path.exists(temp_extract_dir):
    shutil.rmtree(temp_extract_dir)

print("Dados extraídos e organizados com sucesso dentro de '/dataset'!")

## 2. Ingestão dos CSVs Brutos para Tabelas Delta (Bronze)

**Objetivo:** Ler os arquivos CSV brutos extraídos do SISCAN/DATASUS e persisti-los como tabelas em formato **Delta Lake** no catálogo `tcc-analise-siscan-10crs.bronze`.

### Diretrizes de Engenharia de Dados Aplicadas na Camada Bronze:
1. **Fidelidade à Fonte (*Raw Data*):** Os dados são gravados no estado original sem limpezas, renomeações de colunas ou filtros de negócios.
2. **Suporte a Caracteres Especiais (`Column Mapping`):** Uso do parâmetro `delta.columnMapping.mode = "name"` para preservar nomes de colunas originais do SISCAN (com acentos, espaços e pontos) sem falhas no esquema.
3. **Rastreabilidade e Auditoria:** Adição automatizada dos metadados `_ingestion_time` (timestamp da carga) e `_file_source` (caminho do arquivo de origem via `_metadata.file_path`).

---
### 📊 Mapeamento de Tabelas Geradas:
* **`bronze.exames_geral_raw`:** Matriz histórica de quantidade de exames (Q1 e Q2).
* **`bronze.faixa_etaria_raw`:** Distribuição de exames por faixas etárias (Q3).
* **`bronze.adequabilidade_raw`:** Classificação da adequabilidade das amostras (Q4).
* **`bronze.resultados_raw`:** Categorias dos resultados citopatológicos alterados (Q5).
* **`bronze.resultados_categoria_mensal_raw` / `resultados_categoria_anual_raw` / `resultados_categoria_faixa_etaria_raw`:** Resultados por categoria citopatológica (ASC-US, ASC-H, NIC I, carcinoma, glandulares, micro-invasora) — ver seção 2.1.
* **`bronze.resultados_alterados_consolidado_raw`:** Consolidado 2020-2025 de exames alterados por faixa etária — ver seção 2.1.
* **`bronze.outubro_rosa_faixa_etaria_raw`:** Exames mensais (set–dez, 2020-2025) por município × faixa etária — download dedicado ao cruzamento Outubro Rosa × idade — ver seção 2.2.

In [0]:
from pyspark.sql import functions as F

# Base de leitura dos ficheiros brutos
PATH_RAW = "/Volumes/tcc-analise-siscan-10crs/bronze/siscan_files_bronze/dataset"

# 1. Ingestão de Exames Gerais (Q1 e Q2)
df_geral = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"{PATH_RAW}/q1_q2_exames_geral_2020_2025/*.csv")
    .withColumn("_ingestion_time", F.current_timestamp())
    .withColumn("_file_source", F.col("_metadata.file_path"))
)

(
    df_geral.write.format("delta")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .mode("overwrite")
    .saveAsTable("`tcc-analise-siscan-10crs`.bronze.exames_geral_raw")
)

# 2. Ingestão de Faixa Etária (Q3)
df_faixa_etaria = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"{PATH_RAW}/Q3_Faixa_Etaria/*.csv")
    .withColumn("_ingestion_time", F.current_timestamp())
    .withColumn("_file_source", F.col("_metadata.file_path"))
)

(
    df_faixa_etaria.write.format("delta")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .mode("overwrite")
    .saveAsTable("`tcc-analise-siscan-10crs`.bronze.faixa_etaria_raw")
)

# 3. Ingestão de Adequabilidade (Q4)
df_adequabilidade = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"{PATH_RAW}/Q4_Adequabilidade/*.csv")
    .withColumn("_ingestion_time", F.current_timestamp())
    .withColumn("_file_source", F.col("_metadata.file_path"))
)

(
    df_adequabilidade.write.format("delta")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .mode("overwrite")
    .saveAsTable("`tcc-analise-siscan-10crs`.bronze.adequabilidade_raw")
)

# 4. Ingestão de Resultados Citopatológicos (Q5)
df_resultados = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"{PATH_RAW}/Q5_Resultados_Citopatologicos/q5-siscan_colo_alterados_faixa_etaria_*.csv")
    .withColumn("_ingestion_time", F.current_timestamp())
    .withColumn("_file_source", F.col("_metadata.file_path"))
)

(
    df_resultados.write.format("delta")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .mode("overwrite")
    .saveAsTable("`tcc-analise-siscan-10crs`.bronze.resultados_raw")
)

## 2.1. Ingestão dos Datasets Q5 Adicionais — Resultados por Categoria Citopatológica

**Objetivo:** Ingerir os CSVs adicionais do Q5 (extraídos por categoria de resultado citopatológico) em tabelas bronze **separadas por estrutura**, sem misturá-los com a `resultados_raw` original (exames alterados por ano), que permanece intacta e compatível com a camada Silver.

### 📊 Novas Tabelas Bronze:
* **`bronze.resultados_categoria_mensal_raw`:** 2 arquivos com matriz mensal (município × meses 2020-2025) — categorias **ASC-US** e **ASC-H**.
* **`bronze.resultados_categoria_anual_raw`:** 6 arquivos com matriz anual (município × colunas 2020-2025) — **NIC I alto/baixo grau (HPV)**, **carcinoma epidermoide**, **glandulares** (não neoplásicas / não afasta lesão) e **micro-invasora**.
* **`bronze.resultados_categoria_faixa_etaria_raw`:** 8 arquivos, um por categoria, com matriz de faixa etária (município × faixas).
* **`bronze.resultados_alterados_consolidado_raw`:** 1 arquivo consolidado 2020-2025 de exames alterados por faixa etária.

### ⚠️ Diretrizes técnicas:
1. **Cabeçalho próprio por arquivo:** o TABNET omite colunas zeradas — os meses, anos e faixas presentes variam entre arquivos da mesma categoria de estrutura. Por isso a leitura é feita **arquivo a arquivo** e a consolidação usa `unionByName(allowMissingColumns=True)`, alinhando as colunas **pelo nome** e preenchendo com NULL as ausentes (evita desalinhamento posicional).
2. **Encoding ISO-8859-1 (latin-1):** os CSVs adicionais foram exportados em latin-1 — sem esta opção, o `Ç` de `MARÇO` seria corrompido pelo padrão UTF-8.
3. **Classificação automática por estrutura:** a função `estrutura_arquivo` inspeciona o cabeçalho de cada CSV e o classifica como `mensal` (colunas `MES/ANO`) ou `anual` (colunas apenas com o ano), garantindo que a re-execução continue correta mesmo se novos arquivos forem adicionados à pasta.
4. **Fidelidade e auditoria:** sem filtros ou renomeações; mesmos metadados `_ingestion_time` e `_file_source` das demais tabelas bronze.

In [0]:
import os
from functools import reduce

PATH_Q5 = f"{PATH_RAW}/Q5_Resultados_Citopatologicos"

def estrutura_arquivo(path):
    """Classifica o CSV pelo cabeçalho: 'mensal' (colunas MES/ANO) ou 'anual' (colunas só ANO)."""
    with open(path, encoding="latin-1") as fh:
        cols = [c.strip().strip('"') for c in fh.readline().split(";")]
    return "mensal" if any("/" in c for c in cols[1:]) else "anual"

def read_q5_csv(path):
    """Lê um CSV TABNET/SISCAN (latin-1) com cabeçalho próprio + metadados de auditoria."""
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("delimiter", ";")
        .option("encoding", "ISO-8859-1")
        .load(path)
        .withColumn("_ingestion_time", F.current_timestamp())
        .withColumn("_file_source", F.col("_metadata.file_path"))
    )

def ingest_q5_grupo(file_names, table_name):
    """Lê cada arquivo com seu próprio cabeçalho e une por NOME de coluna (colunas ausentes -> NULL)."""
    dfs = [read_q5_csv(f"{PATH_Q5}/{name}") for name in sorted(file_names)]
    df = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)
    (
        df.write.format("delta")
        .option("delta.columnMapping.mode", "name")
        .option("delta.minReaderVersion", "2")
        .option("delta.minWriterVersion", "5")
        .mode("overwrite")
        .saveAsTable(f"`tcc-analise-siscan-10crs`.bronze.{table_name}")
    )
    print(f"{table_name}: {len(file_names)} arquivos -> {df.count()} linhas")

arquivos = sorted([f for f in os.listdir(PATH_Q5) if f.endswith(".csv")])

# 1. Categorias com matriz MENSAL (ASC-H, ASC-US)
mensais = [f for f in arquivos if f.endswith("_anual.csv") and estrutura_arquivo(f"{PATH_Q5}/{f}") == "mensal"]
# 2. Categorias com matriz por ANO (colunas 2020-2025)
anuais = [f for f in arquivos if f.endswith("_anual.csv") and estrutura_arquivo(f"{PATH_Q5}/{f}") == "anual"]
# 3. Categorias por faixa etária
faixas = [f for f in arquivos if f.endswith("_faixa_etaria.csv")]
# 4. Consolidado 2020-2025 de exames alterados
consolidado = [f for f in arquivos if f.startswith("q5-siscan_Exames_Alterados")]

print(f"CSVs na pasta Q5: {len(arquivos)} | mensais: {len(mensais)} | anuais: {len(anuais)} | faixas: {len(faixas)} | consolidado: {len(consolidado)}")

ingest_q5_grupo(mensais, "resultados_categoria_mensal_raw")
ingest_q5_grupo(anuais, "resultados_categoria_anual_raw")
ingest_q5_grupo(faixas, "resultados_categoria_faixa_etaria_raw")
ingest_q5_grupo(consolidado, "resultados_alterados_consolidado_raw")

## 2.2. Ingestão Q02 Outubro Rosa — Exames Mensais (Set–Dez) por Faixa Etária

**Objetivo:** Ingerir os 24 CSVs do download TABNET dedicado ao cruzamento Outubro Rosa × faixa etária (pendência da análise Q1/Q2 × Q3): exames citopatológicos por **município de residência × competência mensal (setembro a dezembro) × faixa etária**, anos 2020–2025 — um arquivo por mês/ano (`Setembr_2020.csv` … `Dezemb_2025.csv`).

### 📊 Nova Tabela Bronze:
* **`bronze.outubro_rosa_faixa_etaria_raw`:** 24 arquivos × 12 linhas (11 municípios + Total) = **288 linhas**; estrutura município × 15 faixas etárias (10–14 a 75–79, "Acima de 79") + Total.

### ⚠️ Características da fonte (mesma pegadinha do Q5):
* **TABNET omite colunas zeradas:** 3 arquivos (`Setembr_2022.csv`, `Novem_2022.csv`, `Setembr_2024 .csv`) não têm a coluna `Entre 10 a 14 anos` (nenhum exame <15 anos na competência) — ler **POR ARQUIVO** e unir com `unionByName(allowMissingColumns=True)`; leitura por glob desalinha os valores uma coluna.
* **Tipos uniformes (`inferSchema=false`):** com inferência por arquivo, a mesma coluna viraria `int` em uns e `string` em outros, causando `DELTA_FAILED_TO_MERGE_FIELDS` na gravação. Tudo como string preserva a fidelidade da fonte.
* **DROP + CREATE:** o `mode("overwrite")` sobre tabela Delta existente tenta fundir schemas; por isso a tabela é dropada antes da recriação (reprocessamento idempotente).
* **Fidelidade:** `SANT\'ANA DO LIVRAMENTO` mantém o escape literal do TABNET (limpeza fica a cargo da Silver); nomes de arquivo com espaço (`Setembr_2024 .csv`, `Setembr_2025 .csv`) são preservados no `_file_source`.

### ✅ Validação (executada em 10/nov/2026):
* Cobertura: 6 anos (2020–2025) × 4 meses (Set/Out/Nov/Dez) ✓ · 24 arquivos, 288 linhas, 19 colunas.
* Fechamento por linha (soma das 15 faixas = coluna Total): **0 divergências** em 288 linhas.
* Fechamento regional por arquivo (linha Total = soma dos 11 municípios): **0 divergências** em 24 arquivos.
* Total regional set–dez 2020–2025: **67.925 exames**.

In [0]:
import os
from functools import reduce

PATH_Q02_OR = f"{PATH_RAW}/Q02-outubroRosa"

def read_q02_or_csv(path):
    """Lê um CSV TABNET (latin-1) com cabeçalho próprio, tudo como string (tipos uniformes entre arquivos)."""
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("delimiter", ";")
        .option("encoding", "ISO-8859-1")
        .load(path)
        .withColumn("_ingestion_time", F.current_timestamp())
        .withColumn("_file_source", F.col("_metadata.file_path"))
    )

arquivos_q02_or = sorted([f for f in os.listdir(PATH_Q02_OR) if f.endswith(".csv")])
print(f"CSVs na pasta Q02-outubroRosa: {len(arquivos_q02_or)}")

# TABNET omite colunas zeradas (ex.: 'Entre 10 a 14 anos') — ler POR ARQUIVO e unir por NOME
dfs_q02_or = [read_q02_or_csv(f"{PATH_Q02_OR}/{name}") for name in arquivos_q02_or]
df_outubro_rosa = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs_q02_or)

# DROP + CREATE: o overwrite sobre tabela Delta existente tenta fundir schemas — drop garante idempotência
spark.sql("DROP TABLE IF EXISTS `tcc-analise-siscan-10crs`.bronze.outubro_rosa_faixa_etaria_raw")

(
    df_outubro_rosa.write.format("delta")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .mode("overwrite")
    .saveAsTable("`tcc-analise-siscan-10crs`.bronze.outubro_rosa_faixa_etaria_raw")
)

print("Outubro Rosa Faixa Etária:", spark.table("`tcc-analise-siscan-10crs`.bronze.outubro_rosa_faixa_etaria_raw").count())

## 3. Validação da Carga e Auditoria de Registros

**Objetivo:** Verificar a persistência física dos dados, realizar contagem de linhas e inspecionar a amostragem das tabelas Delta recém-criadas na Camada Bronze.

In [0]:
# 2. Verificar a quantidade total de registros em cada tabela
print("Exames Geral:", spark.table("`tcc-analise-siscan-10crs`.bronze.exames_geral_raw").count())
print("Faixa Etária:", spark.table("`tcc-analise-siscan-10crs`.bronze.faixa_etaria_raw").count())
print("Adequabilidade:", spark.table("`tcc-analise-siscan-10crs`.bronze.adequabilidade_raw").count())
print("Resultados:", spark.table("`tcc-analise-siscan-10crs`.bronze.resultados_raw").count())
print("Resultados Categoria Mensal:", spark.table("`tcc-analise-siscan-10crs`.bronze.resultados_categoria_mensal_raw").count())
print("Resultados Categoria Anual:", spark.table("`tcc-analise-siscan-10crs`.bronze.resultados_categoria_anual_raw").count())
print("Resultados Categoria Faixa Etária:", spark.table("`tcc-analise-siscan-10crs`.bronze.resultados_categoria_faixa_etaria_raw").count())
print("Resultados Alterados Consolidado:", spark.table("`tcc-analise-siscan-10crs`.bronze.resultados_alterados_consolidado_raw").count())
print("Outubro Rosa Faixa Etária:", spark.table("`tcc-analise-siscan-10crs`.bronze.outubro_rosa_faixa_etaria_raw").count())

### 3.1. Inspeção Visual da Tabela Delta (`exames_geral_raw`)

**Objetivo:** Exibir a amostragem dos dados brutos persistidos na tabela `exames_geral_raw` em formato de planilha interativa.

* **Metadados:** Permite verificar a presença das colunas técnicas `_ingestion_time` e `_file_source` adicionadas no pipeline de ingestão[cite: 1, 15].
* **Estrutura Matricial:** Apresenta as 73 colunas de meses/anos originais do SISCAN/TABNET associadas aos 12 municípios da 10ª CRS[cite: 1, 6, 15].

In [0]:
# 1. Visualizar o conteúdo em formato de planilha interativa
display(spark.table("`tcc-analise-siscan-10crs`.bronze.exames_geral_raw"))